In [287]:
# import numpy as np
import pandas as pd
# import plotly.express as px
# from plotly.subplots import make_subplots
import pandapower as pp # v : 2.14.8

import matplotlib.pyplot as plt
import pp_heig_simulation as pp_sim
import pp_heig_plot as pp_plot
from datetime import time
import numpy as np

import os

# Suppress future warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)


# Import data from pickle

In [288]:
# import pickle file
net_Trey_pickle = pp.from_pickle("trey_net_student.p")

In [289]:
# print buses
net_Trey_pickle.bus

,name,type,zone,in_service
0,STMT003438,PQ,Trafo,True
1,CDBT004764,PQ,North,True
2,CDBT003746,PQ,North,True
3,CDBT004760,PQ,North,True
4,CDBT012139,PQ,North,True
5,CDBT900784,PQ,North,True
6,CDBT901452,PQ,South,True
7,CDBT004774,PQ,South,True
8,CDBT901604,PQ,South,True
9,CDBT016055,PQ,South,True


In [290]:
# print load
net_Trey_pickle.load


,name,bus,const_z_percent,const_i_percent,sn_mva,in_service,type
0,STMT003438,0,0.0,0.0,None,True,wye
1,CDBT004764,1,0.0,0.0,None,True,wye
2,CDBT003746,2,0.0,0.0,None,True,wye
3,CDBT004760,3,0.0,0.0,None,True,wye
4,CDBT012139,4,0.0,0.0,None,True,wye
5,CDBT900784,5,0.0,0.0,None,True,wye
6,CDBT901452,6,0.0,0.0,None,True,wye
7,CDBT004774,7,0.0,0.0,None,True,wye
8,CDBT901604,8,0.0,0.0,None,True,wye
9,CDBT016055,9,0.0,0.0,None,True,wye


In [291]:
net_Trey_pickle.ext_grid

,name,vm_pu,va_degree,slack_weight,s_sc_max_mva,s_sc_min_mva,rx_min,rx_max,r0x0_max,x0x_max,in_service
0,ext_grid_0,1.0,0.0,1.0,100.0,None,None,0.1,2.0,2.0,True


In [292]:
net_Trey_pickle.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service
0,GKN3x150_150,0,1,0.0,1.0,None,True
1,GKN3x150_150_2,1,2,0.0,1.0,None,True
2,GKN3X95_95,1,3,0.0,1.0,None,True
3,GKT3X50_50,3,4,0.0,1.0,None,True
4,GKT3X50_50_2,4,5,0.0,1.0,None,True
5,GKN3x150_150_3,0,6,0.0,1.0,None,True
6,GKN3X95_95_2,6,7,0.0,1.0,None,True
7,GKN3x150_150_4,7,8,0.0,1.0,None,True
8,GKN3X50_50,7,9,0.0,1.0,None,True
9,GKN3X50_50_2,6,10,0.0,1.0,None,True


In [293]:
pd.set_option('display.max_columns', None)
net_Trey_pickle.trafo

,name,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,i0_percent,vector_group,vk0_percent,vkr0_percent,mag0_percent,mag0_rx,si0_hv_partial,shift_degree,tap_side,tap_neutral,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,tap_phase_shifter,parallel,std_type,df,in_service
0,STMT003438,12,0,0.63,18.3,0.42,4.0,0.42,0.65,1.8,Dyn,3.0,0.75,0.3,1.9,1.9,0.0,lv,0,-3,3,2.0,0.0,-3,False,1,None,1.0,True


### Paramètres principaux du transfo
- Puissance nominale : 630 kVA
- Tension MT : 18.3 kV
- Tension BT : 0.42 kV
- Tension de court-circuit : vk = 4 %
- Résistance de court-circuit : vkr = 0.42 %
- Groupe vectoriel : Dyn
- Pertes à vide : 0.65 kW
- Courant à vide : 1.8 %

# Adding missing elements
## Lines

In [294]:
# Distance of lines addition
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3x150_150', 'length_km'] = 0.18
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3x150_150_2', 'length_km'] = 0.1
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3X95_95', 'length_km'] = 0.115
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKT3X50_50', 'length_km'] = 0.08
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKT3X50_50_2', 'length_km'] = 0.09
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3x150_150_3', 'length_km'] = 0.135
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3X95_95_2', 'length_km'] = 0.1
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3x150_150_4', 'length_km'] = 0.205
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3X50_50', 'length_km'] = 0.085
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3X50_50_2', 'length_km'] = 0.255
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'] == 'GKN3X240_240', 'length_km'] = 0.34

# Current capacity of lines addition
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3x150_150', na=False), 'max_i_ka'] = 0.4
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X95_95', na=False), 'max_i_ka'] = 0.252
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains(r'GKN3X50_50|GKT3X50_50', na=False), 'max_i_ka'] = 0.17
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X240_240', na=False), 'max_i_ka'] = 0.512

# Capacity of lines update
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3x150_150', na=False), 'c_nf_per_km'] = 349
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X95_95', na=False), 'c_nf_per_km'] = 338
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains(r'GKN3X50_50|GKT3X50_50', na=False), 'c_nf_per_km'] = 298
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X240_240', na=False), 'c_nf_per_km']  = 346

# Resistance of lines addition
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3x150_150', na=False), 'r_ohm_per_km'] = 0.124
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X95_95', na=False), 'r_ohm_per_km'] = 0.193
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains(r'GKN3X50_50|GKT3X50_50', na=False), 'r_ohm_per_km'] = 0.387
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X240_240', na=False), 'r_ohm_per_km'] = 0.0754

# Reactance of lines addition
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3x150_150', na=False), 'x_ohm_per_km'] = 0.07
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X95_95', na=False), 'x_ohm_per_km'] = 0.07
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains(r'GKN3X50_50|GKT3X50_50', na=False), 'x_ohm_per_km'] = 0.07
net_Trey_pickle.line.loc[net_Trey_pickle.line['name'].str.contains('GKN3X240_240', na=False), 'x_ohm_per_km'] = 0.07

net_Trey_pickle.line['parallel'] = 1  

net_Trey_pickle.line

,name,from_bus,to_bus,g_us_per_km,df,std_type,in_service,length_km,max_i_ka,c_nf_per_km,r_ohm_per_km,x_ohm_per_km,parallel
0,GKN3x150_150,0,1,0.0,1.0,None,True,0.180,0.400,349.0,0.1240,0.07,1
1,GKN3x150_150_2,1,2,0.0,1.0,None,True,0.100,0.400,349.0,0.1240,0.07,1
2,GKN3X95_95,1,3,0.0,1.0,None,True,0.115,0.252,338.0,0.1930,0.07,1
3,GKT3X50_50,3,4,0.0,1.0,None,True,0.080,0.170,298.0,0.3870,0.07,1
4,GKT3X50_50_2,4,5,0.0,1.0,None,True,0.090,0.170,298.0,0.3870,0.07,1
5,GKN3x150_150_3,0,6,0.0,1.0,None,True,0.135,0.400,349.0,0.1240,0.07,1
6,GKN3X95_95_2,6,7,0.0,1.0,None,True,0.100,0.252,338.0,0.1930,0.07,1
7,GKN3x150_150_4,7,8,0.0,1.0,None,True,0.205,0.400,349.0,0.1240,0.07,1
8,GKN3X50_50,7,9,0.0,1.0,None,True,0.085,0.170,298.0,0.3870,0.07,1
9,GKN3X50_50_2,6,10,0.0,1.0,None,True,0.255,0.170,298.0,0.3870,0.07,1


In [295]:
Smax_GKN3x50_50 = np.sqrt(3) * 0.4 * 0.17  # MVA
print("Smax_GKN3x50_50:", Smax_GKN3x50_50)

Smax_GKN3x50_50: 0.11777945491468367


## Loads

In [296]:
# Active power addition for all loads
net_Trey_pickle.load['p_mw'] = 0.004  # 4 kW per house

# Reactive power addition for all loads (cos phi = 0.95 -> q = p * tan(acos(0.95)) = 1.32 kVar)
net_Trey_pickle.load['q_mvar'] = 0.00132

# Constant impedance/current shares for all loads (percent)
net_Trey_pickle.load['const_z_percent'] = 20
net_Trey_pickle.load['const_i_percent'] = 30

# Scaling for all loads
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'STMT003438', 'scaling'] = 5 # For 5 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT004764', 'scaling'] = 8 # For 8 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT003746', 'scaling'] = 3 # For 3 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT004760', 'scaling'] = 4 # For 4 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT012139', 'scaling'] = 4 # For 4 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT900784', 'scaling'] = 4 # For 4 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT901452', 'scaling'] = 5 # For 5 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT004774', 'scaling'] = 4 # For 4 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT901604', 'scaling'] = 1 # For 1 house
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'CDBT016055', 'scaling'] = 4 # For 4 houses
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == 'N1', 'scaling'] = 1 # For 1 house
net_Trey_pickle.load.loc[net_Trey_pickle.load['name'] == '60437', 'scaling'] = 1 # For 1 house

net_Trey_pickle.load

,name,bus,const_z_percent,const_i_percent,sn_mva,in_service,type,p_mw,q_mvar,scaling
0,STMT003438,0,20,30,None,True,wye,0.004,0.00132,5.0
1,CDBT004764,1,20,30,None,True,wye,0.004,0.00132,8.0
2,CDBT003746,2,20,30,None,True,wye,0.004,0.00132,3.0
3,CDBT004760,3,20,30,None,True,wye,0.004,0.00132,4.0
4,CDBT012139,4,20,30,None,True,wye,0.004,0.00132,4.0
5,CDBT900784,5,20,30,None,True,wye,0.004,0.00132,4.0
6,CDBT901452,6,20,30,None,True,wye,0.004,0.00132,5.0
7,CDBT004774,7,20,30,None,True,wye,0.004,0.00132,4.0
8,CDBT901604,8,20,30,None,True,wye,0.004,0.00132,1.0
9,CDBT016055,9,20,30,None,True,wye,0.004,0.00132,4.0


## Ext grid

In [297]:
# Add external grid bus 12 (power entry point)
net_Trey_pickle.ext_grid.loc[net_Trey_pickle.ext_grid['name'] == 'ext_grid_0', 'bus'] = 12

net_Trey_pickle.ext_grid

,name,vm_pu,va_degree,slack_weight,s_sc_max_mva,s_sc_min_mva,rx_min,rx_max,r0x0_max,x0x_max,in_service,bus
0,ext_grid_0,1.0,0.0,1.0,100.0,None,None,0.1,2.0,2.0,True,12.0


## Bus

In [298]:
# Adding vn_kv to bus
net_Trey_pickle.bus['vn_kv'] = 0.42
net_Trey_pickle.bus.loc[net_Trey_pickle.bus['name'] == 'STMT003438HV', 'vn_kv'] = 18.3

net_Trey_pickle.bus

,name,type,zone,in_service,vn_kv
0,STMT003438,PQ,Trafo,True,0.42
1,CDBT004764,PQ,North,True,0.42
2,CDBT003746,PQ,North,True,0.42
3,CDBT004760,PQ,North,True,0.42
4,CDBT012139,PQ,North,True,0.42
5,CDBT900784,PQ,North,True,0.42
6,CDBT901452,PQ,South,True,0.42
7,CDBT004774,PQ,South,True,0.42
8,CDBT901604,PQ,South,True,0.42
9,CDBT016055,PQ,South,True,0.42


## Grid checking

In [299]:
net_Trey_pickle

This pandapower network includes the following parameter tables:
   - bus (13 elements)
   - load (12 elements)
   - ext_grid (1 element)
   - line (11 elements)
   - trafo (1 element)

In [300]:
pp_plot.plot_power_network(
    net=net_Trey_pickle,
    plot_title="net_Trey_pickle",
    filename="net_Trey_pickled",
)

## Simulation

In [301]:
# Safe fix: set rows with missing bus out of service, then cast bus -> int and in_service -> bool
for name, df in list(net_Trey_pickle.items()):
    if isinstance(df, pd.DataFrame) and 'bus' in df.columns:
        # mark rows with missing bus as out of service if in_service exists
        if 'in_service' in df.columns:
            df.loc[df['bus'].isna(), 'in_service'] = False
        # drop rows that still have no valid bus (optional)
        df = df.dropna(subset=['bus'])
        # cast bus to integer numpy type
        df['bus'] = df['bus'].astype(int)
        # ensure in_service is boolean
        if 'in_service' in df.columns:
            df['in_service'] = df['in_service'].astype(bool)
        net_Trey_pickle[name] = df

In [302]:
pp.runpp(net_Trey_pickle)

# matrice d'impédance nodale
pd.DataFrame(net_Trey_pickle._ppc["internal"]["Ybus"].toarray()).round(1)



,0,1,2,3,4,5,6,7,8,9,10,11,12
0,19.6-29.1j,-6.0+ 3.4j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,-8.0+ 4.5j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,-3.7+3.4j,-1.8+16.7j
1,-6.0+ 3.4j,23.8-12.0j,-10.8+ 6.1j,-7.0+ 2.5j,0.0+ 0.0j,0.0+0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
2,0.0+ 0.0j,-10.8+ 6.1j,10.8- 6.1j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
3,0.0+ 0.0j,-7.0+ 2.5j,0.0+ 0.0j,12.5- 3.5j,-5.5+ 1.0j,0.0+0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
4,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,-5.5+ 1.0j,10.4- 1.9j,-4.9+0.9j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
5,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,-4.9+ 0.9j,4.9-0.9j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
6,-8.0+ 4.5j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,17.8- 7.8j,-8.1+ 2.9j,0.0+0.0j,0.0+0.0j,-1.7+0.3j,0.0+0.0j,0.0+ 0.0j
7,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,-8.1+ 2.9j,18.5- 6.8j,-5.3+3.0j,-5.2+0.9j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
8,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+ 0.0j,-5.3+ 3.0j,5.3-3.0j,0.0+0.0j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j
9,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+ 0.0j,0.0+0.0j,0.0+ 0.0j,-5.2+ 0.9j,0.0+0.0j,5.2-0.9j,0.0+0.0j,0.0+0.0j,0.0+ 0.0j


In [303]:
# Valeur des bus
net_Trey_pickle.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,0.935202,-0.592037,0.019110,0.006306
1,0.920980,-0.764557,0.030270,0.009989
2,0.919951,-0.777206,0.011343,0.003743
3,0.914001,-0.777612,0.015060,0.004970
4,0.907853,-0.723502,0.014995,0.004948
5,0.904392,-0.692776,0.014958,0.004936
6,0.928273,-0.676046,0.019016,0.006275
7,0.923746,-0.684302,0.015165,0.005004
8,0.923044,-0.692923,0.003789,0.001250
9,0.920498,-0.655959,0.015130,0.004993


In [304]:
# Puissance totale du réseaux extérieur
net_Trey_pickle.res_ext_grid

,p_mw,q_mvar
0,0.16984,0.069428


In [305]:
pp_plot.plot_powerflow_result(
    net=net_Trey_pickle,
    plot_title="Result Power Flow Trey Net",
    filename="Result_Power_Flow_Trey_Net",
)


---
---
---
# TIMESERIES - TIMESERIES - TIMESERIES
---
---
---


In [306]:
net_Trey_pickle.load.insert(1, "profile_mapping", [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

# add Sgen to grid
pp.create_sgen(
    net_Trey_pickle,
    bus=10,
    p_mw=-0.0,
    name="N1",
)
pp.create_sgen(
    net_Trey_pickle,
    bus=11,
    p_mw=-0.0,
    name="60437",
)

# Ajouter profile_mapping
net_Trey_pickle.sgen["profile_mapping"] = net_Trey_pickle.sgen["bus"]

---
## 1. Saturday summer with production and bride
---

In [307]:
net_Trey_pickle.trafo
net_Trey_pickle.trafo['vn_lv_kv'] = 0.441


### Paramètres principaux du transfo
- Puissance nominale : 630 kVA
- Tension MT : 18.3 kV
- Tension BT : 0.42 kV // + 0.5% = 0.441 // - 0.5% = 0.399
- Tension de court-circuit : vk = 4 %
- Résistance de court-circuit : vkr = 0.42 %
- Groupe vectoriel : Dyn
- Pertes à vide : 0.65 kW
- Courant à vide : 1.8 %

In [308]:
profile_file_path = "input-data/power_profile_P_Q_summer_saturday_with_prod_200kW.xlsx"
output_folder = ""
output_file_name = "power_profile_P_Q_summer_saturday_with_prod_200kW_gradin"

# Charge excel file powerprofile
time_series: dict = pp_sim.load_power_profile_form_xlsx(file_path=profile_file_path)


# Apply power profile to equipement
for eq in ["load","sgen"]:
    pp_sim.apply_power_profile(net=net_Trey_pickle, equipment=eq, power_profiles=time_series[eq])


pp_sim.create_output_writer(net=net_Trey_pickle)

# Run time simulation
result_df = pp_sim.run_time_simulation(
    net=net_Trey_pickle, output_filename=output_file_name
)

# Nom de la charge extrait du nom de fichier
load_name = os.path.basename(profile_file_path).replace("power_profile_P_Q_", "").replace(".xlsx", "")

# Lecture du fichier Excel avec 2 lignes d'en-tête
df = pd.read_excel(profile_file_path, header=[0, 1])

df_load = pd.read_excel(
    profile_file_path,
    sheet_name="load",
    header=[0, 1]
)

p_load = pd.DataFrame(index=df_load.iloc[:, 0])  # Time

for i in range(12):
    p_col = df_load.iloc[:, 1 + i*2]  # P [MW]
    p_load[f"Load_{i}"] = p_col.values * 1000
 
df_sgen = pd.read_excel(
    profile_file_path,
    sheet_name="sgen",
    header=[0, 1]
)

p_sgen = pd.DataFrame(index=df_sgen.iloc[:, 0])  # Time

for i in range(12):
    p_col = df_sgen.iloc[:, 1 + i*2]  # P [MW]
    p_sgen[f"Sgen_{i}"] = p_col.values * 1000



### Power profiles, Bus voltage, Line loading, Powerflow

In [309]:
pp_plot.plot_timeseries_result(
    data_df=p_sgen,
    ylabel="P [kW]",
    plot_title=f"Power profiles of all gen ({load_name})",
    filename=os.path.join(output_file_name + "_power_profiles_all_loads")
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_bus.vm_pu"],
    ylabel="V [pu]",
    plot_title=f"Bus voltage for ({load_name})",
    filename=os.path.join(output_file_name + "_voltage_result")
)

pp_plot.plot_timeseries_result(
    data_df=result_df["res_line.loading_percent"],
    ylabel="[%]",
    plot_title=f"Line loading for ({load_name})",
    filename=os.path.join(output_file_name + "_line_loading_result"),
)

for plot_time in [time(hour=12, minute=45)]:

    pp_plot.plot_timestamps_powerflow_result(
        net=net_Trey_pickle,
        plot_time=plot_time,
        filename=os.path.join(output_file_name + "_power_flow_at_" + pp_plot._time_to_str(plot_time)),
    )
    print()